# 07 Build All-Candidate Final Decision Audit Table
This notebook joins deterministic decisions with LLM tie-breaker outputs for every candidate pair, assigns final pair-level decisions, marks which candidates are selected or dropped, and writes an audit table covering both selected and non-selected candidates.

In [0]:
# Load deterministic decisions and cached LLM tie-breaker results.
from pyspark.sql import functions as F

decisions_table = "workspace.entity_resolution_project.company_er_decisions"
llm_table = "workspace.entity_resolution_project.company_er_llm_tiebreaker"

all_candidate_decisions_table = "workspace.entity_resolution_project.company_er_all_candidate_decisions"

LLM_CONFIDENCE_THRESHOLD = 0.70

decisions = spark.table(decisions_table)

llm = (
    spark.table(llm_table)
    .select(
        "left_row_key",
        "right_row_key",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "llm_error_message",
        "llm_response_clean"
    )
    .dropDuplicates(["left_row_key", "right_row_key"])
)

In [0]:
# Combine deterministic and LLM results into final all-candidate decisions.
all_candidate_decisions = (
    decisions
    .join(
        llm,
        on=["left_row_key", "right_row_key"],
        how="left"
    )
    .withColumn(
        "final_decision",
        F.when(F.col("decision") == "MATCH", F.lit("MATCH"))
         .when(F.col("decision") == "NO_MATCH", F.lit("NO_MATCH"))
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision") == "MATCH") &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("MATCH")
         )
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision") == "NO_MATCH") &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("NO_MATCH")
         )
         .otherwise(F.lit("REVIEW"))
    )
    .withColumn(
        "final_decision_source",
        F.when(F.col("decision").isin("MATCH", "NO_MATCH"), F.lit("deterministic"))
         .when(
             (F.col("decision") == "AMBIGUOUS") &
             (F.col("llm_decision").isin("MATCH", "NO_MATCH")) &
             (F.col("llm_confidence") >= F.lit(LLM_CONFIDENCE_THRESHOLD)),
             F.lit("llm_tiebreaker")
         )
         .otherwise(F.lit("manual_review"))
    )
    .withColumn(
        "final_confidence",
        F.when(F.col("final_decision_source") == "llm_tiebreaker", F.col("llm_confidence"))
         .otherwise(F.col("composite_score"))
    )
    .withColumn("is_top_candidate", F.col("candidate_rank") == 1)
    .withColumn(
        "is_selected_final_candidate",
        F.col("candidate_rank") == 1
    )
    .withColumn(
        "final_selection_status",
        F.when(F.col("is_selected_final_candidate"), F.lit("SELECTED_FOR_FINAL_DECISION"))
         .otherwise(F.lit("NOT_SELECTED_FOR_FINAL_DECISION"))
    )
    .withColumn(
        "dropped_stage",
        F.when(
            ~F.col("is_selected_final_candidate"),
            F.lit("final_selection_top_candidate_filter")
        )
    )
    .withColumn(
        "needs_human_review",
        F.col("final_decision") == "REVIEW"
    )
    .withColumn(
        "resolved_company_name",
        F.when(
            (F.col("is_selected_final_candidate") == True) &
            (F.col("final_decision") == "MATCH"),
            F.col("right_company_name")
        )
    )
    .withColumn(
        "resolved_company_key",
        F.when(
            (F.col("is_selected_final_candidate") == True) &
            (F.col("final_decision") == "MATCH"),
            F.col("right_row_key")
        )
    )
    .withColumn(
        "final_reason",
        F.concat_ws(
            " | ",
            F.concat(F.lit("candidate_pair_decision="), F.col("decision")),
            F.concat(F.lit("deterministic_rule="), F.col("decision_rule")),
            F.concat(F.lit("candidate_rank="), F.col("candidate_rank").cast("string")),
            F.concat(F.lit("selected_final_candidate="), F.col("is_selected_final_candidate").cast("string")),
            F.concat(F.lit("llm_decision="), F.coalesce(F.col("llm_decision"), F.lit("none"))),
            F.concat(
                F.lit("llm_confidence="),
                F.coalesce(F.round(F.col("llm_confidence"), 3).cast("string"), F.lit("none"))
            ),
            F.concat(F.lit("llm_reason="), F.coalesce(F.col("llm_reason"), F.lit("none"))),
            F.concat(F.lit("score="), F.round(F.col("composite_score"), 3).cast("string")),
            F.concat(F.lit("top1="), F.round(F.col("top1_score"), 3).cast("string")),
            F.concat(F.lit("top2="), F.round(F.col("top2_score"), 3).cast("string")),
            F.concat(F.lit("gap="), F.round(F.col("score_gap_top1_top2"), 3).cast("string")),
            F.concat(F.lit("entropy_norm="), F.round(F.col("entropy_norm"), 3).cast("string")),
            F.concat(F.lit("name_similarity="), F.round(F.col("name_similarity"), 3).cast("string")),
            F.concat(F.lit("semantic_similarity="), F.round(F.col("semantic_similarity"), 3).cast("string")),
            F.concat(F.lit("country_match="), F.col("country_match").cast("string")),
            F.concat(F.lit("city_match="), F.col("city_match").cast("string"))
        )
    )
    .withColumn(
        "selection_reason",
        F.when(
            F.col("is_selected_final_candidate"),
            F.lit("This candidate is rank 1 for the input company and is kept for the final decision table.")
        )
        .otherwise(
            F.concat(
                F.lit("This candidate is not rank 1 for the input company, so it is excluded from company_er_final_decisions. Candidate rank: "),
                F.col("candidate_rank").cast("string"),
                F.lit(". Pair-level decision before final selection: "),
                F.col("final_decision"),
                F.lit(".")
            )
        )
    )
)

In [0]:
# Save the all-candidate decision audit table and print row counts.
(
    all_candidate_decisions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(all_candidate_decisions_table)
)

print("All candidate decision rows:", all_candidate_decisions.count())
print("Unique input records:", all_candidate_decisions.select("left_row_key").distinct().count())
print(
    "Selected final candidate rows:",
    all_candidate_decisions.filter(F.col("is_selected_final_candidate")).count()
)
print(
    "Not selected candidate rows:",
    all_candidate_decisions.filter(~F.col("is_selected_final_candidate")).count()
)

In [0]:
# Summarize final decisions by source and selection status.
display(
    all_candidate_decisions
    .groupBy("final_decision", "final_decision_source", "final_selection_status")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# Review all candidate decisions in ranked order.
display(
    all_candidate_decisions
    .select(
        "left_row_key",
        "left_company_name",
        "right_company_name",
        "candidate_rank",
        "is_selected_final_candidate",
        "final_selection_status",
        "dropped_stage",
        "final_decision",
        "final_decision_source",
        "final_confidence",
        "top1_score",
        "top2_score",
        "score_gap_top1_top2",
        "name_similarity",
        "semantic_similarity",
        "country_match",
        "city_match",
        "llm_decision",
        "llm_confidence",
        "llm_reason",
        "selection_reason",
        "final_reason"
    )
    .orderBy("left_row_key", "candidate_rank")
)

In [0]:
# Inspect candidates dropped from the final top-candidate table.
display(
    all_candidate_decisions
    .filter(~F.col("is_selected_final_candidate"))
    .select(
        "left_company_name",
        "right_company_name",
        "candidate_rank",
        "final_decision",
        "final_confidence",
        "decision_rule",
        "selection_reason",
        "final_reason"
    )
    .orderBy("left_company_name", "candidate_rank")
)